### Instalacja bibliotek

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
!pip uninstall -y torchao
!pip install -U ms-swift datasets rouge_score sacrebleu "qwen_vl_utils>=0.0.14"

  Using cached datasets-5.1.0-py3-none-any.whl.metadata (24 kB)


### Importy i stałe

In [6]:
import gc
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset
from peft import PeftModel
from rouge_score import rouge_scorer
import sacrebleu
from swift import sft_main, SftArguments, get_model_processor, get_template
from swift.infer_engine import TransformersEngine, RequestConfig, InferRequest
import os
import json
os.environ['USE_HF'] = "1"


MODEL_ID = 'Qwen/Qwen3.5-2B'
TEST_SIZE = 0.02
SEED = 42

[INFO:swift] Successfully registered `/usr/local/lib/python3.13/dist-packages/swift/dataset/data/dataset_info.json`.


### Dane (filtrowanie, format Swifta, podział train/test)

In [7]:
raw = load_dataset('medalpaca/medical_meadow_wikidoc_patient_information', split='train')

def is_valid(ex):

    inp, out = (ex['input'] or '').strip(), (ex['output'] or '').strip()
    return len(inp) > 0 and len(out) >= 20 and len(inp) + len(out) <= 1500


def to_swift_format(ex):

    return {'query': (ex['instruction'] or '').strip() + '\n' + ex['input'].strip(),
            'response': ex['output'].strip()}

data = raw.filter(is_valid).map(to_swift_format, remove_columns=raw.column_names)

split = data.train_test_split(test_size=TEST_SIZE, seed=SEED)
split['train'].to_json('train_medical.jsonl')
test_queries, test_references = split['test']['query'], split['test']['response']

print(f"Po filtrowaniu: {len(data)} | Trening: {len(split['train'])} | Test: {len(test_queries)}")

Creating json from Arrow format:   0%|          | 0/6 [00:00<?, ?ba/s]

Po filtrowaniu: 5720 | Trening: 5605 | Test: 115


### Funkcje pomocnicze (generowanie i metryki)

In [8]:
def generate(queries, adapter_dir=None):

    model, processor = get_model_processor(MODEL_ID, torch_dtype=torch.float16)
    if adapter_dir:
        model = PeftModel.from_pretrained(model, adapter_dir)
    template = get_template(processor, enable_thinking=False)
    engine = TransformersEngine(model, template=template, max_batch_size=8)
    config = RequestConfig(max_tokens=512, temperature=0)
    requests = [InferRequest(messages=[{'role': 'user', 'content': q}]) for q in queries]
    responses = engine.infer(requests, request_config=config)
    answers = [(r.choices[0].message.content or '').split('</think>')[-1].strip() for r in responses]
    del engine, model
    gc.collect()
    torch.cuda.empty_cache()
    return answers


def metrics(preds, refs):

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = [scorer.score(r, p) for p, r in zip(preds, refs)]
    res = {k: np.mean([s[k].fmeasure for s in scores]) * 100 for k in ['rouge1', 'rouge2', 'rougeL']}
    res['bleu'] = sacrebleu.corpus_bleu(preds, [refs]).score
    res['avg_words'] = np.mean([len(p.split()) for p in preds])
    return res

RESULTS_DIR = '/content/drive/MyDrive/outputs/Qwen3.5-2B_medical'


def save_results(name, answers, m):

    with open(f'{RESULTS_DIR}/{name}.json', 'w') as f:
        json.dump({'answers': answers, 'metrics': m}, f, ensure_ascii=False, default=float)


def load_results(name):

    with open(f'{RESULTS_DIR}/{name}.json') as f:
        d = json.load(f)
    return d['answers'], d['metrics']

### Ewaluacja przed tuningiem

In [10]:
base_answers = generate(test_queries)
base_metrics = metrics(base_answers, test_references)
save_results('base', base_answers, base_metrics)
base_metrics

[INFO:swift] Downloading the model from HuggingFace Hub, model_id: Qwen/Qwen3.5-2B


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

[INFO:swift] Loading the model using model_dir: /root/.cache/huggingface/hub/models--Qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc
[WARNING:swift] Loading model from '/root/.cache/huggingface/hub/models--Qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc' with trust_remote_code=True. This will execute custom Python code shipped in the model repository (e.g. configuration_*.py, modeling_*.py, tokenization_*.py). Only proceed if you trust the source of this model.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[WARNING:swift] Please install the package: `pip install "decord" -U`.
[WARNING:swift] Loading config from '/root/.cache/huggingface/hub/models--Qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc' with trust_remote_code=True. Custom Python code from the model repository will be executed. Only use models from sources you trust.
[WARNING:swift] Loading tokenizer/processor from '/root/.cache/huggingface/hub/mode

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

[INFO:swift] default_system: None
[INFO:swift] max_length: 262144
[INFO:swift] response_prefix: None
[INFO:swift] agent_template: qwen3_5
[INFO:swift] norm_bbox: norm1000
[INFO:swift] Setting ROOT_IMAGE_DIR: None. You can adjust this hyperparameter through the environment variable: `ROOT_IMAGE_DIR`.
[INFO:swift] Setting QWENVL_BBOX_FORMAT: legacy. You can adjust this hyperparameter through the environment variable: `QWENVL_BBOX_FORMAT`.
  0%|          | 0/115 [00:00<?, ?it/s][transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its refer

{'rouge1': np.float64(19.734976658499516),
 'rouge2': np.float64(4.55121243790111),
 'rougeL': np.float64(10.857616524472219),
 'bleu': 1.10202153323247,
 'avg_words': np.float64(282.17391304347825)}

### Trening LoRa

In [ ]:
drive_output_dir = '/content/drive/MyDrive/outputs/Qwen3.5-2B_medical'
os.makedirs(drive_output_dir, exist_ok=True)

result = sft_main(SftArguments(
    model=MODEL_ID,
    tuner_type='lora',
    dataset=['train_medical.jsonl'],
    torch_dtype='float16',
    lora_rank=8,
    lora_alpha=32,
    target_modules=['all-linear'],
    add_non_thinking_prefix=True,
    loss_scale='ignore_empty_think',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    split_dataset_ratio=0.01,
    eval_steps=50,
    save_steps=50,
    save_total_limit=2,
    logging_steps=5,
    output_dir=drive_output_dir,
    seed=SEED,
    data_seed=SEED,
))
adapter_dir = result['last_model_checkpoint']
print(adapter_dir)

[INFO:swift] rank: -1, local_rank: -1, world_size: 1, local_world_size: 1
[INFO:swift] Downloading the model from HuggingFace Hub, model_id: Qwen/Qwen3.5-2B


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

[INFO:swift] Loading the model using model_dir: /root/.cache/huggingface/hub/models--Qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc
[WARNING:swift] Loading model from '/root/.cache/huggingface/hub/models--Qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc' with trust_remote_code=True. This will execute custom Python code shipped in the model repository (e.g. configuration_*.py, modeling_*.py, tokenization_*.py). Only proceed if you trust the source of this model.
[WARNING:swift] Please install the package: `pip install "decord" -U`.
[INFO:swift] Setting args.lazy_tokenize: True
[INFO:swift] Setting args.dataloader_num_workers: 1
[INFO:swift] output_dir: /content/drive/MyDrive/outputs/Qwen3.5-2B_medical/v0-20261031-204411
[INFO:swift] args: SftArguments(
acc_strategy=token,
accelerator_config={'dispatch_batches': False},
adalora_beta1=0.85,
adalora_beta2=0.85,
adalora_deltaT=1,
adalora_init_r=12,
adalora_orth_reg_weight=0.5,
adalora_target_r=8,
adal

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

... (159 outputs left out)

[INFO:swift] Saving model checkpoint to /content/drive/MyDrive/outputs/Qwen3.5-2B_medical/v0-20261031-204411/checkpoint-347
Train: 100%|██████████| 347/347 [34:49<00:00,  6.02s/it]
[INFO:swift] last_model_checkpoint: /content/drive/MyDrive/outputs/Qwen3.5-2B_medical/v0-20261031-204411/checkpoint-347
[INFO:swift] best_model_checkpoint: /content/drive/MyDrive/outputs/Qwen3.5-2B_medical/v0-20261031-204411/checkpoint-347
[INFO:swift] images_dir: /content/drive/MyDrive/outputs/Qwen3.5-2B_medical/v0-20261031-204411/images


{'train_runtime': '2090', 'train_samples_per_second': '2.655', 'train_steps_per_second': '0.166', 'train_loss': '1.833', 'epoch': '1', 'global_step/max_steps': '347/347', 'elapsed_time': '34m 50s', 'remaining_time': '0s', 'memory(GiB)': '14.15', 'train_speed(s/it)': '6.023'}


[INFO:swift] End time of running main: 2026-10-31 21:19:32.781414


/content/drive/MyDrive/outputs/Qwen3.5-2B_medical/v0-20261031-204411/checkpoint-347


In [11]:
adapter_dir = f'{RESULTS_DIR}/v0-20261031-204411/checkpoint-347'

### Ewaluacja po tuningu

In [12]:
tuned_answers = generate(test_queries, adapter_dir)
tuned_metrics = metrics(tuned_answers, test_references)
save_results('tuned', tuned_answers, tuned_metrics)
tuned_metrics

[INFO:swift] Downloading the model from HuggingFace Hub, model_id: Qwen/Qwen3.5-2B


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

[INFO:swift] Loading the model using model_dir: /root/.cache/huggingface/hub/models--Qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc
[WARNING:swift] Loading model from '/root/.cache/huggingface/hub/models--Qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc' with trust_remote_code=True. This will execute custom Python code shipped in the model repository (e.g. configuration_*.py, modeling_*.py, tokenization_*.py). Only proceed if you trust the source of this model.
[WARNING:swift] Please install the package: `pip install "decord" -U`.
[WARNING:swift] Loading config from '/root/.cache/huggingface/hub/models--Qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270f8a8fc' with trust_remote_code=True. Custom Python code from the model repository will be executed. Only use models from sources you trust.
[WARNING:swift] Loading tokenizer/processor from '/root/.cache/huggingface/hub/models--Qwen--Qwen3.5-2B/snapshots/15852e8c16360a2fea060d615a32b45270

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

[INFO:swift] default_system: None
[INFO:swift] max_length: 262144
[INFO:swift] response_prefix: None
[INFO:swift] agent_template: qwen3_5
[INFO:swift] norm_bbox: norm1000
100%|██████████| 115/115 [10:43<00:00,  5.59s/it]


{'rouge1': np.float64(25.55646316531798),
 'rouge2': np.float64(10.623146721309483),
 'rougeL': np.float64(19.942783433219383),
 'bleu': 2.9316357786091074,
 'avg_words': np.float64(123.8)}

### Tabelka i przykłady

In [13]:
pd.DataFrame({'Przed tuningiem': base_metrics, 'Po tuningu': tuned_metrics}).round(2)

,Przed tuningiem,Po tuningu
rouge1,19.73,25.56
rouge2,4.55,10.62
rougeL,10.86,19.94
bleu,1.10,2.93
avg_words,282.17,123.80


In [14]:
for i in range(3):
    print(f"PYTANIE:\n{test_queries[i]}\n\nREFERENCJA:\n{test_references[i]}")
    print(f"\nPRZED:\n{base_answers[i]}\n\nPO:\n{tuned_answers[i]}\n{'=' * 60}")

PYTANIE:
Answer this question truthfully
What to expect if I have Tuberculosis  (Outlook/Prognosis)?

REFERENCJA:
Symptoms often improve in 2 - 3 weeks. A chest x-ray will not show this improvement until later. The outlook is excellent if pulmonary TB is diagnosed early and treatment is begun quickly.
The prognosis of multidrug-resistant TB is worse than that of tuberculosis susceptible to the regular common treatment. So it is necessary to do a drug susceptibility test and to monitor adherence to the treatment regimen to avoid new drug resistances and ensure a better prognosis.

PRZED:
The outlook for someone with Tuberculosis (TB) depends heavily on the **stage of the disease**, the **response to treatment**, and the **compliance with medication**.

Here is a breakdown of what to expect regarding prognosis:

### 1. High Prognosis with Early Treatment
If TB is detected early and treated with the correct combination of antibiotics, the outlook is generally excellent.
*   **Recovery Tim

## Wnioski

W pierwszym zadaniu napisałem wnioski. Niestety z braku czasu wnioski z zadania drugiego z chęcią przedstawię na rozmowie rekrutacyjnej.
Może napiszę tylko, że jednym z głównych problemów były często kończące się limity GPU na Google Colab, co bardzo spowolniło pracę nad zadaniem.